# Gene deletion prevalence for hrp2/3
This notebook is designed to import and visualise point mutation data aggregated by `nomadic summarize`. The notebook can be run one cell at a time (Shift-Enter) or all together ('Run All' above).

In [ ]:
import sys
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
from statsmodels.stats.proportion import proportion_confint

sys.path.append("../functions")
from gene_deletions import DeletionFinder
from workspace import Workspace

# Settings


In [ ]:
# Decide whether you want the outputs to be saved and in which format
save_results = True
save_format = "svg"

# Load workspace
ws = Workspace()

# Define where the outputs will be saved
output_dir = Path.cwd() / "results" / ws.name

if save_results:
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"All results will be saved to: {output_dir}")

# Functions

In [ ]:
# Copied from nomadic verbatim, except gene_deletions_df join changed from right to left
def gene_deletion_prevalence_by(
    gene_deletions_df: pd.DataFrame, master_df: pd.DataFrame, fields: list[str]
) -> pd.DataFrame:
    """
    Compute the prevalence of gene deletions in `gene_deletions_df`
    stratified by columns in `fields`.
    """
    gene_deletions_df = gene_deletions_df.merge(
        master_df[["sample_id", *fields]], on="sample_id", how="right"
    )

    prev_df = (
        gene_deletions_df.groupby(["gene", *fields])
        .agg(
            n_samples=pd.NamedAgg("is_deleted", len),
            n_passed=pd.NamedAgg("is_deleted", lambda x: sum(x.notnull())),
            n_deleted=pd.NamedAgg("is_deleted", lambda x: sum(x)),
        )
        .reset_index()
    )

    # Compute prevalence
    prev_df["prevalence"] = 100 * prev_df["n_deleted"] / prev_df["n_passed"]

    # Compute prevalence 95% confidence intervals
    low, high = proportion_confint(
        prev_df["n_deleted"],
        prev_df["n_passed"],
        alpha=0.05,
        method="beta",
    )
    prev_df["prevalence_lowci"] = 100 * low
    prev_df["prevalence_highci"] = 100 * high

    return prev_df

In [ ]:
def generate_deletion_prevalence_barchart(
    deletions_df: pd.DataFrame,
    master_df: pd.DataFrame,
    by: Optional[str] = "All",
    fig_prefix: str = None,
    min_count: int = None,
) -> go.Figure:
    """
    Build a barchart from the df provided
    """
    if min_count is not None and by == "All":
        raise ValueError("min_count can only be used with a grouping variable")

    if by == "All":
        plot_df = gene_deletion_prevalence_by(deletions_df, master_df, [])
    else:
        plot_df = gene_deletion_prevalence_by(deletions_df, master_df, [by])
    
    if min_count is not None:
        plot_df = plot_df[plot_df["n_passed"] >= min_count]

    genes = set(plot_df["gene"])

    fig_name = f"Prevalence of {', '.join(genes)} deletions"

    if fig_prefix is not None:
        fig_name = f"{fig_prefix}: {fig_name}"


    data = []
    htemp = "%{y:0.1f}% (%{customdata[2]}/%{customdata[1]})"
    
    if by == "All":
        # Prepare plotting data
        customdata = np.stack(
            [
                plot_df["n_samples"],
                plot_df["n_passed"],
                plot_df["n_deleted"],
            ],
            axis=-1,
        )
        data.append(
            go.Bar(
                x=plot_df["gene"],
                y=plot_df["prevalence"],
                customdata=customdata,
                hovertemplate=htemp,
                name="Prevalence",
                error_y=dict(
                    type="data",
                    array=plot_df["prevalence_highci"] - plot_df["prevalence"],
                    arrayminus=plot_df["prevalence"]
                    - plot_df["prevalence_lowci"],
                ),
            )
        )
    else:
        for group in plot_df[by].unique():
            group_df = plot_df.query(f"{by} == @group")
            # Prepare plotting data
            customdata = np.stack(
                [
                    group_df["n_samples"],
                    group_df["n_passed"],
                    group_df["n_deleted"],
                ],
                axis=-1,
            )
            data.append(
                go.Bar(
                    x=group_df["gene"],
                    y=group_df["prevalence"],
                    customdata=customdata,
                    hovertemplate=htemp,
                    name=str(group),
                    error_y=dict(
                        type="data",
                        array=group_df["prevalence_highci"] - group_df["prevalence"],
                        arrayminus=group_df["prevalence"]
                        - group_df["prevalence_lowci"],
                    ),
                )
            )
        fig_name = (f"{fig_name} by {by.capitalize()}")

        if min_count is not None:
            fig_name += f" (min n={min_count})"

    # Plotting
    fig = go.Figure(data)
    fig.update_layout(
        yaxis_title="Prevalence (%)",
        xaxis=dict(showline=True, linewidth=1, linecolor="black", mirror=True),
        yaxis=dict(
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            showgrid=True,
            gridcolor="lightgray",
            gridwidth=0.5,
            griddash="dot",
        ),
        legend=dict(
            orientation="h", yanchor="top", y=-0.15, xanchor="left", x=0
        ),
        plot_bgcolor="rgba(0,0,0,0)",
        hovermode="x unified",
    )
    fig.update_yaxes(range=[0, 100])
    fig.update_traces(marker=dict(line=dict(color="black", width=1)))
    fig.update_layout(title=fig_name)
    
    if save_results:
        fig.write_image(
            output_dir / f"{fig_name.replace(' ', '_')}.{save_format}",
        )
    return fig

In [ ]:
def munge_model_outputs (df: pd.DataFrame) -> pd.DataFrame:
    long_df = df.melt(
        id_vars=['sample_id','sample_type'],
        value_vars=['hrp2', 'hrp3'],
        var_name='gene',
        value_name='is_deleted'
    )
    return long_df.groupby(['sample_id', 'sample_type', 'gene'])['is_deleted'].agg(
        n_deleted='sum', n_replicates='count', is_deleted='max').reset_index()

# Prepare data

### Load and filter

In [ ]:
master_df = pd.read_csv(ws.master_csv_path)
qc_cov = pd.read_csv(ws.summaries_path / "summary.replicates_qc.csv")

In [ ]:
# This code removes samples failing QC
dfs = []

for res_dir in ws.results_path.iterdir():
    print(f"Processing {res_dir.name}")
    cov_df = pd.read_csv(res_dir / "summary.region_coverage.csv")

    # Identify barcodes that have failed QC (only samples are in the qc file)
    qc_exp = qc_cov[qc_cov["expt_name"] == res_dir.name]
    failed_bcs = list(qc_exp["barcode"][~qc_exp["passing"]].unique())
    passed_bcs = set(qc_exp["barcode"][qc_exp["passing"]])
    if len(passed_bcs) == 0:
            print(f"WARNING: No samples passed QC for {res_dir.name}. Skipping...")
            continue
    print(f"   Dropped {len(failed_bcs)} samples that failed QC")

    # Add back in the controls - negatives are critical
    exp_meta = pd.read_csv(res_dir / "metadata" / "samples.csv")
    if "sample_type" not in exp_meta.columns:
        print(f"WARNING: No sample_type column identified. Skipping {res_dir.name}...")
        continue
    pos_bcs = set(exp_meta["barcode"][exp_meta["sample_type"].str.lower().isin(["pos","positive"])])
    neg_bcs = set(exp_meta["barcode"][exp_meta["sample_type"].str.lower().isin(["neg","negative"])])
    passed_bcs = passed_bcs | pos_bcs
    if len(neg_bcs) == 0:
        print(f"WARNING: No negative controls identified. Skipping {res_dir.name}...")
        continue

    # Get final filtered df
    cov_df_filtered = cov_df[cov_df["barcode"].isin(passed_bcs | neg_bcs)].copy()
    
    del_cls = DeletionFinder(cov_df_filtered)        
    del_cls.estimate_hyperparameters(negative_barcodes=list(neg_bcs))
    
    for gene in del_cls.deleted_amplicons:
        gene_short = gene.split("-")[0]
        print(f"Processing for {gene_short}")
        del_cls.run_mcmc(target_gene=gene)
    summary = del_cls.summarise_mcmc_outputs()
    summary["expt_name"] = res_dir.name
    # Join in sample_type
    summary = summary.merge(exp_meta, on="barcode")
    dfs.append(summary)
        
if len(dfs) == 0:
    print("No valid experiments identified")
else:
    deletions_df = pd.concat(dfs, ignore_index=True)
    # Transform into expected output
    all_del_df = deletions_df.rename(columns={"hrp2_del_prediction": "hrp2", "hrp3_del_prediction": "hrp3"})

In [ ]:
final_del_df = munge_model_outputs(all_del_df)
if save_results:
    final_del_df.to_csv(output_dir / "gene_deletions_prediction.csv", index=False)

# Plots

In [ ]:
generate_deletion_prevalence_barchart(final_del_df, master_df)

In [ ]:
final_del_df[final_del_df["is_deleted"]]

# Misc plots

In [ ]:
ax = sns.stripplot(data = del_cls.df_bedcov, x="barcode", y="n_reads", hue="name", log_scale=True)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
ax = sns.stripplot(data = del_cls.df_bedcov, x="name", y="n_reads", hue="name", log_scale=True)
ax.axhline(100, color="red", linestyle="--", linewidth=1)
plt.xticks(rotation=90)
plt.show()